# City Fixed Effects

**DS4DH · Module 05 — Regression Analysis**

*Technique:* Dummy variables, the dummy trap, and controlling for city without estimating its cost

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/05b_fixed_effects.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import statsmodels.api as sm

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

The objection to Model A is concrete:

> Vancouver is expensive for everyone. If immigrants are more likely to live in
> Vancouver and Toronto than in Edmonton, your "penalty" might just be a "lives in
> an expensive city" effect wearing a different name. Strip out the city.

Fixed effects strip it out. You add one indicator per city, and the immigrant
coefficient is then estimated *within* cities rather than across them.

In [ ]:
csd = df.dropna(subset=['csd_code'])

reg_df = csd[csd['immigrant_status'].isin(['Immigrant', 'Non-immigrants'])
             & csd['cma'].isin(CITIES)].dropna(subset=['Renter']).copy()
reg_df['is_immigrant'] = (reg_df['immigrant_status'] == 'Immigrant').astype(int)

print(f'{len(reg_df)} rows — one per (CSD, immigrant status) with a renter STIR')
print(reg_df['immigrant_status'].value_counts().to_string())

In [ ]:
# Do the groups actually sit in different cities? If not, there is nothing
# for the control to do.
share = pd.crosstab(reg_df['cma'], reg_df['immigrant_status'], normalize='columns')
print((share * 100).round(1).to_string())
print()
print('Columns are the share of each group living in each city. Where these')
print('differ, city composition can masquerade as a group effect.')

## Building the dummies

One column per city, 1 where the row belongs to it. But you must drop one —
including all four alongside an intercept makes the columns linearly dependent
(they sum to 1, which is the intercept), and the model cannot be estimated. That
is the **dummy variable trap**.

The dropped city becomes the **reference**: every other city's coefficient is a
difference from it.

In [ ]:
city_dummies = pd.get_dummies(reg_df['cma'], drop_first=True, dtype=float)

print('dummy columns kept:', list(city_dummies.columns))
dropped = [c for c in CITIES if c not in city_dummies.columns]
print('reference city (dropped):', dropped)
print()
print('drop_first drops the alphabetically first category — here Edmonton,')
print('not Montréal. Always print this rather than assuming it.')

In [ ]:
y = reg_df['Renter']
X_a = sm.add_constant(reg_df[['is_immigrant']])
X_b = sm.add_constant(pd.concat([reg_df[['is_immigrant']], city_dummies], axis=1))

model_a = sm.OLS(y, X_a).fit()
model_b = sm.OLS(y, X_b).fit()

print(model_b.summary().tables[1])

In [ ]:
# The comparison that answers the reviewer.
ca = model_a.params['is_immigrant']
cb = model_b.params['is_immigrant']

print(f'Model A  (no controls)   is_immigrant = {ca:+.3f} pp   p={model_a.pvalues["is_immigrant"]:.4f}')
print(f'Model B  (city FE)       is_immigrant = {cb:+.3f} pp   p={model_b.pvalues["is_immigrant"]:.4f}')
print()
print(f'shrinkage: {ca:+.3f} -> {cb:+.3f}  ({(1 - cb / ca):.0%} of the gap was city composition)')
print()
print(f'R-squared  A: {model_a.rsquared:.4f}   B: {model_b.rsquared:.4f}')
print('City explains far more of the variation than immigrant status does.')

## What the shrinkage means

Roughly 30% of the raw gap was city composition rather than a group difference.
The remainder is the within-city gap — and it is still not distinguishable from
zero.

Notice what fixed effects did *not* require: you never had to know, or estimate
correctly, how much more expensive Vancouver is. The dummies absorb whatever the
city-level difference happens to be, including causes you never thought of, as
long as they are constant within a city.

### 🔧 Your turn 1

Refit Model B using `drop_first=False` and manually dropping `Vancouver` instead:

```python
d2 = pd.get_dummies(reg_df['cma'], dtype=float).drop(columns=['Vancouver'])
```

The city coefficients all change. Does `is_immigrant` change? What does that tell
you about which numbers in a fixed-effects table are interpretable?

In [ ]:
# What the city coefficients say, relative to the reference.
ref = dropped[0]
print(f'Renter STIR relative to {ref}:')
print()
for c in city_dummies.columns:
    print(f'  {c:<12}{model_b.params[c]:>+8.2f} pp   p={model_b.pvalues[c]:.4f}')
print()
print(f'  {ref:<12}{0.0:>+8.2f} pp   (reference, by construction)')

### 🔧 Your turn 2

Add a size control alongside the city dummies — `np.log10(reg_df['rent_pop'])`,
after dropping rows where it is missing.

Does `is_immigrant` move again? Each control you add answers a different version
of the reviewer's objection; the coefficient means something slightly different
each time.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Every city coefficient changes, because each is a difference
from whichever city you dropped. `is_immigrant` does not move at all — it is
invariant to the choice of reference. That is the rule: in a fixed-effects model,
the coefficient on your variable of interest is interpretable, and the fixed
effects themselves are only interpretable relative to an arbitrary baseline. Do
not put them in a policy brief.

**Your turn 2.** Adding log renter population moves `is_immigrant` again, usually
slightly. Every control changes the question: Model A asks "do immigrant renters
spend more?", Model B asks "within the same city, do they?", and the third asks
"within the same city and among similarly sized rental markets, do they?". These
are three different questions and the honest report names which one it answered.

</details>

## Where this stops

The coefficient is now defensible. Its standard error is not — the model still
assumes the variance is constant across cities, and it plainly is not. That is
notebook 05c.